In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import pandas as pd
#data collection
Final_data = pd.read_csv(r"C:\Users\chaka\Preethu\My_Git_Repo\Ecotype project 3\Final_data.csv")
Final_data

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type,Wilderness_Area,Soil_Type
0,2596,51,3,258.0,0,510,221.0,232,148,6279,Aspen,1,29
1,2590,56,2,212.0,-6,390,220.0,235,151,6225,Aspen,1,29
2,2804,139,9,268.0,65,3180,234.0,238,135,6121,Lodgepole Pine,1,12
3,2785,155,18,242.0,117,3090,238.0,238,122,6211,Lodgepole Pine,1,30
4,2595,45,2,153.0,-1,391,220.0,234,150,6172,Aspen,1,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145885,2834,88,8,376.0,44,2552,232.0,227,128,1595,Lodgepole Pine,1,29
145886,2832,68,4,390.0,44,2522,224.0,231,142,1572,Lodgepole Pine,1,29
145887,2829,80,7,390.0,33,2492,229.0,228,133,1550,Lodgepole Pine,1,29
145888,2826,121,7,379.0,30,2462,232.0,234,135,1528,Lodgepole Pine,1,29


In [4]:
Final_data['Cover_Type'].value_counts() #multiclass imbalance

Cover_Type
Lodgepole Pine       103071
Spruce/Fir            31110
Aspen                  3069
Krummholz              2160
Ponderosa Pine         2160
Douglas-fir            2160
Cottonwood/Willow      2160
Name: count, dtype: int64

 Class Imbalance Handling:
 

In [5]:
from imblearn.over_sampling import SMOTE, SMOTENC

In [6]:
#feature definition
x = Final_data.drop("Cover_Type", axis=1) #feature columns
y = Final_data["Cover_Type"]              #target columns

In [7]:
#applying SMOTE
smote = SMOTE(random_state=42)
x_resampled, y_resampled = smote.fit_resample(x, y)
#df conversion
df_resampled = pd.DataFrame(x_resampled, columns=x.columns)
df_resampled['Cover_Type']= y_resampled
df_resampled['Cover_Type'].value_counts() # Check class distribution

Cover_Type
Aspen                103071
Lodgepole Pine       103071
Spruce/Fir           103071
Krummholz            103071
Ponderosa Pine       103071
Douglas-fir          103071
Cottonwood/Willow    103071
Name: count, dtype: int64

In [8]:
df_resampled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 721497 entries, 0 to 721496
Data columns (total 13 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Elevation                           721497 non-null  int64  
 1   Aspect                              721497 non-null  int64  
 2   Slope                               721497 non-null  int64  
 3   Horizontal_Distance_To_Hydrology    721497 non-null  float64
 4   Vertical_Distance_To_Hydrology      721497 non-null  int64  
 5   Horizontal_Distance_To_Roadways     721497 non-null  int64  
 6   Hillshade_9am                       721497 non-null  float64
 7   Hillshade_Noon                      721497 non-null  int64  
 8   Hillshade_3pm                       721497 non-null  int64  
 9   Horizontal_Distance_To_Fire_Points  721497 non-null  int64  
 10  Wilderness_Area                     721497 non-null  int64  
 11  Soil_Type                 

In [9]:
df_resampled.to_csv("Sampled_file.csv",index = False)

In [27]:
# Model building
import warnings
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

In [46]:
#train_test_split

X = df_resampled.drop("Cover_Type", axis=1) #feature columns
Y = df_resampled["Cover_Type"]   #target column
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

In [12]:
#Logistic Regression
lor = LogisticRegression()
lor.fit(X_train,Y_train)
Y_pred = lor.predict(X_test) 
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Logistic Regression: Accuracy = {accuracy:.4f}")

Logistic Regression: Accuracy = 0.5303


In [13]:
#Decision Tree Classifier
dtc = DecisionTreeClassifier()
dtc.fit(X_train,Y_train)
Y_pred = dtc.predict(X_test) 
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Decision Tree Classifier: Accuracy = {accuracy:.4f}")

Decision Tree Classifier: Accuracy = 0.9869


In [14]:
#Random Forest Classifier
rfc = RandomForestClassifier()
rfc.fit(X_train,Y_train)
Y_pred = rfc.predict(X_test) 
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Random Forest Classifier: Accuracy = {accuracy:.4f}")

Random Forest Classifier: Accuracy = 0.9938


In [15]:
#K - Neighbors Classifier
knn = KNeighborsClassifier()
knn.fit(X_train,Y_train)
Y_pred = knn.predict(X_test) 
accuracy = accuracy_score(Y_test, Y_pred)
print(f"K-Neighbors Classifier: Accuracy = {accuracy:.4f}")

K-Neighbors Classifier: Accuracy = 0.9914


#XGBoost requires numeric class labels, so encoded the categorical target using LabelEncoder before training.


In [16]:
#XGBoost
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y_encoded,
    test_size=0.2,
    stratify=Y_encoded,
    random_state=42
)
xgb = XGBClassifier(
    objective='multi:softprob',        # multiclass
    num_class=7,   # number of classes
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='mlogloss',
    random_state=42
)

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=Y_train
)
xgb.fit(X_train, Y_train, sample_weight=sample_weights)

Y_pred = xgb.predict(X_test)

accuracy = accuracy_score(Y_test, Y_pred)
print(f"XGBoost Accuracy: {accuracy:.4f}")


XGBoost Accuracy: 0.9630


In [ ]:
#Cross Validation 
from sklearn.model_selection import KFold, cross_validate
skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=42)
#train_test_split
X = df_resampled.drop("Cover_Type", axis=1) #feature columns
Y = df_resampled["Cover_Type"]   #target column

scoring =  scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro'
}
#Model selected for validation
#Random Forest Classifier
rfc = RandomForestClassifier(n_estimators=100, random_state=42)

results = cross_validate(rfc, X, Y, cv=skf, scoring= scoring, return_train_score=False)
print("===== Random Forest 5-Fold Cross Validation Results =====\n")

print("Accuracy :  %.4f (+/- %.4f)" % (
    np.mean(cv_results['test_accuracy']),
    np.std(cv_results['test_accuracy'])
))

print("Precision:  %.4f (+/- %.4f)" % (
    np.mean(cv_results['test_precision']),
    np.std(cv_results['test_precision'])
))

print("Recall   :  %.4f (+/- %.4f)" % (
    np.mean(cv_results['test_recall']),
    np.std(cv_results['test_recall'])
))

print("F1 Score :  %.4f (+/- %.4f)" % (
    np.mean(cv_results['test_f1']),
    np.std(cv_results['test_f1'])
))

===== Random Forest  5-Fold Cross Validation Results =====

Accuracy :  0.9938 (+/- 0.0002)
Precision:  0.9938 (+/- 0.0002)
Recall   :  0.9938 (+/- 0.0002)
F1 Score :  0.9938 (+/- 0.0002)


In [36]:
#We see that among the models we trained, score of Random forest is higher compared to allother models with 99.3% accuracy. 
#The same was cross validates used KFOLD method and model was tested. We get good results.
# Hyper parameter tuning can increase the accuracy score 
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3,4],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}
#model selection
rfc = RandomForestClassifier(
    class_weight='balanced',   # handle imbalance
    random_state=42
)
grid = GridSearchCV(
    estimator=rfc,
    param_grid=param_grid,
    cv=skf,
    scoring='f1_macro',   # very important
    n_jobs=-1,
    verbose=2
)

grid.fit(X, Y)

print("Best Parameters:\n", grid.best_params_)
print("\nBest Macro F1 Score:", round(grid.best_score_, 4))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters:
 {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

Best Macro F1 Score: 0.7286


In [47]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3,4],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rfc = RandomForestClassifier()
skf = StratifiedKFold(
    n_splits=3,      # reduce folds for large data
    shuffle=True,
    random_state=42
)
random_search = RandomizedSearchCV(
    estimator=rfc,
    param_distributions=param_dist,
    n_iter=20,                 # only 20 random combinations
    cv=skf,
    scoring='f1_macro',        # very important
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train, Y_train)
print("Best Parameters:\n", random_search.best_params_)
print("\nBest Macro F1 Score:", round(random_search.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Parameters:
 {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 4}

Best Macro F1 Score: 0.7266


In [48]:
Prediction = random_search.predict(X_test)

In [51]:
import pickle
file = open('rf_cover_type.pkl', 'wb')
pickle.dump(random_search, file)
print("Model saved successfully!")

Model saved successfully!
